In [111]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import yaml
CONFIG_PATH = "../config.yaml"

In [89]:
gold = pl.read_parquet("../data/raw/Gold_1d.parquet")
nifty = pl.read_parquet("../data/raw/Nifty50_1d.parquet")
usdinr = pl.read_parquet("../data/raw/USDINR_1d.parquet")

In [90]:
nifty = nifty.rename({"('Close', '^NSEI')":"Close","('High', '^NSEI')":"High","('Low', '^NSEI')":"Low","('Open', '^NSEI')":"Open","('Volume', '^NSEI')":"Volume"})
gold = gold.rename({"('Close', 'GOLDBEES.NS')":"Close","('High', 'GOLDBEES.NS')":"High","('Low', 'GOLDBEES.NS')":"Low","('Open', 'GOLDBEES.NS')":"Open","('Volume', 'GOLDBEES.NS')":"Volume"})
usdinr = usdinr.rename({"('Close', 'USDINR=X')":"Close","('High', 'USDINR=X')":"High","('Low', 'USDINR=X')":"Low","('Open', 'USDINR=X')":"Open","('Volume', 'USDINR=X')":"Volume"})


In [91]:
bad_dates = [
    pl.datetime(2019, 12, 19),
    pl.datetime(2019, 12, 20),
]
for date in bad_dates:
    nifty = nifty.filter(pl.col("Date") != date)
    gold = gold.filter(pl.col("Date") != date)
    usdinr = usdinr.filter(pl.col("Date") != date)


In [98]:
nifty = nifty.with_columns(
    pl.col("Close").pct_change().alias("Ret_1d"),
    pl.col("Close").pct_change(n=3).alias("Ret_3d"),
    pl.col("Close").pct_change(n=5).alias("Ret_5d"),
    pl.col("Close").pct_change(n=20).alias("Ret_20d"),
    pl.col("Close").rolling_mean(window_size=5).alias("MA_5d"),
    pl.col("Close").rolling_mean(window_size=20).alias("MA_20d"),
    (
        (pl.col("Close").shift(-4) - pl.col("Open").shift(-1)) 
        / pl.col("Open").shift(-1)
    ).alias("Forward_Return")

)
nifty = nifty.with_columns(
    pl.col("Ret_1d").rolling_std(window_size=5).alias("Vol_5d"),
    pl.col("Ret_1d").rolling_std(window_size=10).alias("Vol_10d"),
    pl.col("Ret_1d").rolling_std(window_size=20).alias("Vol_20d"),
)
nifty = nifty.with_columns(
    pl.col("Ret_1d").abs().alias("Abs_Return"),
    pl.col("Ret_1d").abs().rolling_mean(window_size=5).alias("Rolling_Abs_Return"),
    (pl.col("Vol_5d")/pl.col("Vol_20d")).alias("Vol_Ratio"),
    (pl.col("MA_5d")/pl.col("MA_20d")).alias("MA_Ratio"),
    (pl.when((pl.col("High")-pl.col("Low")) !=0)
        .then((pl.col("Close")-pl.col("Low"))/(pl.col("High")-pl.col("Low")))
        .otherwise(0.5)
        .alias("Close_Pos_Range")),
    ((pl.col("Close")-pl.col("Open"))/pl.col("Open")).alias("Intraday_Return"),
    ((pl.col("High")-pl.col("Low"))/pl.col("Close")).rolling_mean(window_size=5).alias("Rolling_Range"),
    (pl.col("Forward_Return") > 0).cast(pl.Int8).alias("Label")
)

nifty = nifty.drop(["Ret_1d","MA_5d","MA_20d","Forward_Return"])

In [99]:
gold = gold.with_columns(
    pl.col("Close").pct_change().alias("Ret_1d"),
    pl.col("Close").pct_change(n=3).alias("Ret_3d"),
    pl.col("Close").pct_change(n=5).alias("Ret_5d"),
    pl.col("Close").pct_change(n=20).alias("Ret_20d"),
    pl.col("Close").rolling_mean(window_size=5).alias("MA_5d"),
    pl.col("Close").rolling_mean(window_size=20).alias("MA_20d"),
    (
        (pl.col("Close").shift(-4) - pl.col("Open").shift(-1)) 
        / pl.col("Open").shift(-1)
    ).alias("Forward_Return")
)
gold = gold.with_columns(
    pl.col("Ret_1d").rolling_std(window_size=5).alias("Vol_5d"),
    pl.col("Ret_1d").rolling_std(window_size=10).alias("Vol_10d"),
    pl.col("Ret_1d").rolling_std(window_size=20).alias("Vol_20d"),
)
gold = gold.with_columns(
    pl.col("Ret_1d").abs().alias("Abs_Return"),
    pl.col("Ret_1d").abs().rolling_mean(window_size=5).alias("Rolling_Abs_Return"),
    (pl.col("Vol_5d")/(pl.col("Vol_20d")+1e-8)).alias("Vol_Ratio"),
    (pl.col("MA_5d")/(pl.col("MA_20d")+1e-8)).alias("MA_Ratio"),
    (pl.when((pl.col("High")-pl.col("Low")) !=0)
        .then((pl.col("Close")-pl.col("Low"))/(pl.col("High")-pl.col("Low")))
        .otherwise(0.5)
        .alias("Close_Pos_Range")),
    ((pl.col("Close")-pl.col("Open"))/pl.col("Open")).alias("Intraday_Return"),
    ((pl.col("High")-pl.col("Low"))/pl.col("Close")).rolling_mean(window_size=5).alias("Rolling_Range"),
    (pl.col("Forward_Return") > 0).cast(pl.Int8).alias("Label")

)


gold = gold.drop(["Ret_1d","MA_5d","MA_20d","Forward_Return"])


In [100]:
usdinr = usdinr.with_columns(
    pl.col("Close").pct_change().alias("Ret_1d"),
    pl.col("Close").pct_change(n=3).alias("Ret_3d"),
    pl.col("Close").pct_change(n=5).alias("Ret_5d"),
    pl.col("Close").pct_change(n=20).alias("Ret_20d"),
    pl.col("Close").rolling_mean(window_size=5).alias("MA_5d"),
    pl.col("Close").rolling_mean(window_size=20).alias("MA_20d"),
    (
        (pl.col("Close").shift(-4) - pl.col("Open").shift(-1)) 
        / pl.col("Open").shift(-1)
    ).alias("Forward_Return")
)
usdinr = usdinr.with_columns(
    pl.col("Ret_1d").rolling_std(window_size=5).alias("Vol_5d"),
    pl.col("Ret_1d").rolling_std(window_size=10).alias("Vol_10d"),
    pl.col("Ret_1d").rolling_std(window_size=20).alias("Vol_20d"),
)
usdinr = usdinr.with_columns(
    pl.col("Ret_1d").abs().alias("Abs_Return"),
    pl.col("Ret_1d").abs().rolling_mean(window_size=5).alias("Rolling_Abs_Return"),
    (pl.col("Vol_5d")/pl.col("Vol_20d")).alias("Vol_Ratio"),
    (pl.col("MA_5d")/pl.col("MA_20d")).alias("MA_Ratio"),
    (pl.when((pl.col("High")-pl.col("Low")) !=0)
        .then((pl.col("Close")-pl.col("Low"))/(pl.col("High")-pl.col("Low")))
        .otherwise(0.5)
        .alias("Close_Pos_Range")),
    ((pl.col("Close")-pl.col("Open"))/pl.col("Open")).alias("Intraday_Return"),
    ((pl.col("High")-pl.col("Low"))/pl.col("Close")).rolling_mean(window_size=5).alias("Rolling_Range"),
    (pl.col("Forward_Return") > 0).cast(pl.Int8).alias("Label")
)

usdinr = usdinr.drop(["Ret_1d","MA_5d","MA_20d","Forward_Return"])

usdinr.columns

['Close',
 'High',
 'Low',
 'Open',
 'Volume',
 'Date',
 'Ret_3d',
 'Ret_5d',
 'Ret_20d',
 'Vol_5d',
 'Vol_10d',
 'Vol_20d',
 'Abs_Return',
 'Rolling_Abs_Return',
 'Vol_Ratio',
 'MA_Ratio',
 'Close_Pos_Range',
 'Intraday_Return',
 'Rolling_Range',
 'Label']

In [110]:
exposure_features = ["Ret_3d","Ret_5d","Ret_20d","Vol_10d","Vol_Ratio","MA_Ratio","Close_Pos_Range","Intraday_Return",]
regime_features = ["Vol_10d","Vol_20d","Vol_Ratio","Abs_Return","Rolling_Abs_Return","Rolling_Range",]

In [112]:
with open(CONFIG_PATH,"r") as f:
    config = yaml.safe_load(f)

config["Exposure_Featuers"] = exposure_features
config["Regime_Featuers"] = regime_features

with open(CONFIG_PATH, "w") as f:
    yaml.dump(config,f,default_flow_style=False)

In [101]:
temporal_split = [
    [pl.datetime(2013,12,31),pl.datetime(2024,1,1)],
    [pl.datetime(2024,2,29),pl.datetime(2026,1,1)]
    ]
nifty_train = nifty.filter(
    (pl.col("Date")>temporal_split[0][0]) & (pl.col("Date")<temporal_split[0][1])
    )
nifty_test = nifty.filter(
    (pl.col("Date")>temporal_split[1][0]) & (pl.col("Date")<temporal_split[1][1])
    )

gold_train = gold.filter(
    (pl.col("Date")>temporal_split[0][0]) & (pl.col("Date")<temporal_split[0][1])
    )
gold_test = gold.filter(
    (pl.col("Date")>temporal_split[1][0]) & (pl.col("Date")<temporal_split[1][1])
    )
usdinr_train = usdinr.filter(
    (pl.col("Date")>temporal_split[0][0]) & (pl.col("Date")<temporal_split[0][1])
    )
usdinr_test = usdinr.filter(
    (pl.col("Date")>temporal_split[1][0]) & (pl.col("Date")<temporal_split[1][1])
    )

In [102]:
nifty_test.head()

Close,High,Low,Open,Volume,Date,Ret_3d,Ret_5d,Ret_20d,Vol_5d,Vol_10d,Vol_20d,Abs_Return,Rolling_Abs_Return,Vol_Ratio,MA_Ratio,Close_Pos_Range,Intraday_Return,Rolling_Range,Label
f64,f64,f64,f64,i64,datetime[ns],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8
22338.75,22353.300781,22047.75,22048.300781,351500,2024-03-01 00:00:00,0.006325,0.005675,0.022191,0.010124,0.007589,0.006649,0.016192,0.00726,1.522674,1.006123,0.952379,0.013173,0.009752,1
22405.599609,22440.900391,22358.300781,22403.5,298800,2024-03-04 00:00:00,0.020703,0.012817,0.029116,0.009691,0.007568,0.006559,0.002993,0.007042,1.477538,1.00725,0.572628,0.000094,0.009341,0
22356.300781,22416.900391,22269.150391,22371.25,296200,2024-03-05 00:00:00,0.016991,0.007115,0.019467,0.009892,0.007603,0.006459,0.0022,0.006793,1.531499,1.007708,0.58985,-0.000668,0.009468,1
22474.050781,22497.199219,22224.349609,22327.5,312300,2024-03-06 00:00:00,0.006057,0.023821,0.024785,0.006956,0.007274,0.006525,0.005267,0.005619,1.066076,1.011207,0.91516,0.006564,0.009042,0
22493.550781,22525.650391,22430.0,22505.300781,379900,2024-03-07 00:00:00,0.003925,0.023234,0.035712,0.007028,0.007019,0.005999,0.000868,0.005504,1.171538,1.014054,0.664407,-0.000522,0.008073,0


In [115]:
nifty_train.write_parquet("../data/processed/train/nifty.parquet")
gold_train.write_parquet("../data/processed/train/gold.parquet")
usdinr_train.write_parquet("../data/processed/train/usdinr.parquet")
nifty_test.write_parquet("../data/processed/test/nifty.parquet")
gold_test.write_parquet("../data/processed/test/gold.parquet")
usdinr_test.write_parquet("../data/processed/test/usdinr.parquet")
